# Part VI — Do Global Characteristics improve distress forecasts?

Run Notebooks 04 and 05 first. This notebook compares the unpenalized CHS benchmark, CHS-only ridge, and ridge with the same eight predictors plus 37 Global Characteristics. Set the input and output paths below if the handoff lives elsewhere.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss, log_loss, precision_recall_curve
from sklearn.preprocessing import StandardScaler
from src.preprocessing import fit_preprocessor, transform_features

ROOT = Path.cwd()
OUT = ROOT / 'outputs' / 'baseline'
manifest = pd.read_csv(ROOT / 'outputs/tables/feature_manifest.csv')
chs = manifest.loc[manifest.selected & manifest.source.eq('CHS'), 'feature'].tolist()
extra = manifest.loc[manifest.selected & manifest.source.eq('Global Characteristics'), 'feature'].tolist()
assert len(chs) == 8 and len(extra) == 37
keys = ['permno', 'forecast_month']
panel = pd.read_parquet(ROOT / 'data/processed/part_III_model_panel.parquet',
                       columns=keys + ['target', 'labeled', 'split'] + chs + extra)
baseline = pd.read_parquet(OUT / 'chs_predictions.parquet')
assert len(baseline) == panel.split.eq('test').sum()

## Model and validation protocol

Fit preprocessing and scaling on initial training rows only. Select C by 1996–2003 validation log loss and choose a separate F1 cutoff for each model on validation.

In [2]:
def matrices(fit_rows, score_rows, features):
    params = fit_preprocessor(fit_rows[features], chs, features[len(chs):])
    x = transform_features(fit_rows[features], params).to_numpy()
    scaler = StandardScaler()
    x = scaler.fit_transform(x)
    z = scaler.transform(transform_features(score_rows[features], params).to_numpy())
    return x, z

def ridge(c, x, y):
    model = LogisticRegression(C=c, max_iter=300, tol=1e-7)
    model.fit(x, y)
    if model.n_iter_[0] >= model.max_iter:
        raise RuntimeError('Ridge did not converge')
    return model

def choose_threshold(y, p):
    precision, recall, cutoffs = precision_recall_curve(y, p)
    f1 = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-15)
    return cutoffs[np.argmax(f1)]

def metrics(y, p, threshold):
    flagged = p >= threshold
    tp, fp = y[flagged].sum(), (1-y[flagged]).sum()
    fn, tn = y[~flagged].sum(), (1-y[~flagged]).sum()
    return dict(roc_auc=roc_auc_score(y,p), average_precision=average_precision_score(y,p),
                log_loss=log_loss(y,p), brier=brier_score_loss(y,p), mean_probability=p.mean(),
                realized=y.mean(), threshold=threshold, precision=tp/max(tp+fp,1),
                recall=tp/(tp+fn), specificity=tn/(tn+fp), f1=2*tp/max(2*tp+fp+fn,1),
                tp=tp, fp=fp, fn=fn, tn=tn)

def develop(features):
    train = panel.loc[panel.labeled & panel.split.eq('train')]
    val = panel.loc[panel.labeled & panel.split.eq('validation')]
    y, y_val = train.target.to_numpy(dtype=int), val.target.to_numpy(dtype=int)
    x, z = matrices(train,val,features)
    result = []
    for c in [0.00001, 0.0001, 0.001, 0.01, 0.1, 1]:
        p = ridge(c,x,y).predict_proba(z)[:,1]
        result.append({'C': c, **metrics(y_val,p,choose_threshold(y_val,p))})
    return pd.DataFrame(result).sort_values(['log_loss', 'C'])

def refit(features, c):
    dev = panel.loc[panel.labeled & panel.split.ne('test')]
    test = panel.loc[panel.split.eq('test')]
    x, z = matrices(dev,test,features)
    p = ridge(c,x,dev.target.to_numpy(dtype=int)).predict_proba(z)[:,1]
    return test[keys].assign(proba=p)

## Freeze the specifications

In [3]:
specs = {'ridge_chs': chs, 'ridge_expanded': chs + extra}
validation = {name: develop(features) for name, features in specs.items()}
choices = {name: grid.iloc[0] for name, grid in validation.items()}
pd.concat(validation, names=['model']).to_csv(OUT / 'part_vi_validation_grid.csv')
display(pd.DataFrame(choices).T[['C','log_loss','average_precision','threshold']])

,C,log_loss,average_precision,threshold
ridge_chs,0.1,0.013816,0.104261,0.069207
ridge_expanded,1.0,0.013729,0.116777,0.060178


## Refit through 2003 and compare on identical test rows

Use the selected C and carry the validation threshold into 2004–2024. The probabilities are evaluated without recalibration.

In [4]:
predictions = baseline.copy()
thresholds = {'chs_logit': pd.read_csv(OUT / 'test_metrics.csv').threshold.iloc[0]}
for name, features in specs.items():
    choice = choices[name]
    forecasts = refit(features, choice.C).rename(columns={'proba': name})
    predictions = predictions.merge(forecasts, on=keys, how='left', validate='one_to_one')
    assert predictions[name].between(0,1).all()
    thresholds[name] = choice.threshold

predictions = predictions.rename(columns={'probability':'chs_logit'})
known = predictions.loc[predictions.target.notna()]
y = known.target.to_numpy(dtype=int)
comparison = pd.DataFrame({name: metrics(y, known[name].to_numpy(), threshold)
                           for name, threshold in thresholds.items()}).T
comparison.to_csv(OUT / 'part_vi_test_comparison.csv', index_label='model')
predictions.to_parquet(OUT / 'part_vi_predictions.parquet', index=False)
display(comparison)

,roc_auc,average_precision,log_loss,brier,mean_probability,realized,threshold,precision,recall,specificity,f1,tp,fp,fn,tn
chs_logit,0.963991,0.065311,0.007011,0.001333,0.002416,0.001353,0.069300,0.071084,0.408989,0.992761,0.121118,546.0,7135.0,789.0,978450.0
ridge_chs,0.964036,0.065453,0.007002,0.001332,0.002407,0.001353,0.069207,0.071635,0.408989,0.992821,0.121916,546.0,7076.0,789.0,978509.0
ridge_expanded,0.963788,0.070139,0.007546,0.001451,0.003357,0.001353,0.060178,0.055292,0.568539,0.986842,0.100783,759.0,12968.0,576.0,972617.0
